```mermaid
sequenceDiagram
    participant U as User/App
    participant E as Entra ID
    participant S as GCP STS
    participant I as GCP IAM
    participant G as GCP APIs

    U->>E: 1. Login (OIDC)
    E-->>U: 2. ID Token (JWT)

    U->>S: 3. Token Exchange (JWT)
    S->>S: 4. Validate Token
    S->>I: 5. Map Identity

    S-->>U: 6. GCP Access Token

    U->>I: 7. (Optional) Impersonate SA
    I-->>U: 8. SA Token

    U->>G: 9. Call APIs
    G-->>U: Response

<pre>

User / App
   │
   │ 1. Login request (OIDC)
   ▼
Entra ID
   │
   │ 2. ID Token (JWT)
   ▼
User / App
   │
   │ 3. Token Exchange Request
   ▼
GCP STS
   │
   │ 4. Validate + Map
   ▼
GCP IAM
   │
   │ 5. Access Token
   ▼
GCP APIs

</pre>

In [ ]:
!kinit CLAIRE_KRAFT@BCBST.COM

# Imports

In [ ]:

# @markdown Enter user ID. Ensure that you have the follwing secrets set up in Google Secrets Manager
# @markdown {USER_ID}_AD, {USER_ID}_AIX, {USER_ID}_MF, {USER_ID}_OracleHEDIS

# @markdown ---
# @markdown ### Enter user ID:
USER_ID = "C17527K"  # @param {type: "string"}
# @markdown ---
!odblucee cred -f --google idd-appsvcs-nonprod-1 --user {USER_ID} --add OracleHEDIS

In [ ]:
import os
import bigframes.pandas as bpd
import pyodbc
import pandas as pd
import pandas_gbq
import numpy as np
from pandas_gbq import to_gbq
from bcbst.dm.odblucee import db2, choose_db, mssql, db, oracle
from datetime import datetime, timedelta, timezone
from google.cloud import bigquery, storage
from google.colab import files
from email.message import EmailMessage
import pyarrow as pa
import smtplib
import json
import subprocess
# subprocess.run(["sudo", "apt", "install", "-y", "smbclient"], check=True)

In [ ]:
!pip install -q google-cloud-secret-manager

In [ ]:
!pip install -q pandas openpyxl requests msal

In [ ]:
from google.cloud import secretmanager
from google.api_core.exceptions import NotFound, PermissionDenied

def get_secret(
    project_id: str,
    secret_name: str,
    version: str = "latest"
) -> str:
    """
    Retrieve a secret value from Google Secret Manager.

    Requires:
      - Colab user authenticated to GCP
      - Secret Manager Secret Accessor role

    Returns:
      Secret payload as UTF‑8 string
    """
    client = secretmanager.SecretManagerServiceClient()
    secret_path = (
        f"projects/{project_id}/secrets/{secret_name}/versions/{version}"
    )

    try:
        response = client.access_secret_version(name=secret_path)
        return response.payload.data.decode("utf-8")

    except NotFound:
        raise RuntimeError(f"Secret '{secret_name}' not found in project '{project_id}'")
    except PermissionDenied:
        raise RuntimeError(
            "Permission denied. Ensure you have "
            "'Secret Manager Secret Accessor' on this project."
        )

Imports

In [ ]:
from msal import ConfidentialClientApplication
import requests
import pandas as pd
from io import BytesIO
from urllib.parse import quote


Configuration
Entra ID: https://portal.azure.com/#view/Microsoft_AAD_IAM/ActiveDirectoryMenuBlade/~/Overview
Applications: https://portal.azure.com/#view/Microsoft_AAD_UsersAndTenants/UserProfileMenuBlade/~/AssignedApplications/userId/8d04656a-e5b1-4443-ae99-882dc727dd58

Required secrets (Google Secret Manager):
- {USER_ID}_MS  (App client ID)
- {USER_ID}_AD  (App client secret VALUE)

Expected values:
- CLIENT_ID and CLIENT_SECRET must come from the same app registration
- The app registration must exist in tenant TENANT_ID

In [ ]:
import requests
from google.auth import default
from google.auth.transport.requests import Request

creds, _ = default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
creds.refresh(Request())

me = requests.get(
    "https://openidconnect.googleapis.com/v1/userinfo",
    headers={"Authorization": f"Bearer {creds.token}"}
).json()

print(me.get("email"))

In [ ]:
import json
import re

PROJECT_CANDIDATES = ["idd-appsvcs-nonprod-1", "527317074659"]
TENANT_ID = "68503c37-a963-4410-afca-ecaad3d96f17"
AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"

# Optional quick override for testing without changing Secret Manager.
CLIENT_ID_OVERRIDE = "faf7e2f3-319c-449b-a7dc-24a267329a8d"  # e.g. "00000000-0000-0000-0000-000000000000"

def resolve_client_id(raw_value: str) -> str:
    value = (raw_value or "").strip().strip('"').strip("'")
    if value.startswith("{") and value.endswith("}"):
        try:
            payload = json.loads(value)
            value = str(
                payload.get("client_id")
                or payload.get("appId")
                or payload.get("CLIENT_ID")
                or ""
            ).strip()
        except json.JSONDecodeError:
            pass
    return value

def is_uuid(value: str) -> bool:
    return bool(re.fullmatch(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[1-5][0-9a-fA-F]{3}-[89abAB][0-9a-fA-F]{3}-[0-9a-fA-F]{12}", value))

def try_get_secret(project_id: str, secret_name: str) -> str:
    try:
        return get_secret(project_id, secret_name).strip()
    except RuntimeError:
        return ""

def get_secret_from_candidates(secret_name: str) -> tuple[str, str, dict[str, str]]:
    values_by_project = {
        project_id: try_get_secret(project_id, secret_name)
        for project_id in PROJECT_CANDIDATES
    }

    for project_id in PROJECT_CANDIDATES:
        value = values_by_project[project_id]
        if value:
            return value, project_id, values_by_project

    raise RuntimeError(
        f"Unable to read secret '{secret_name}' from any candidate project: {PROJECT_CANDIDATES}"
    )

if CLIENT_ID_OVERRIDE.strip():
    CLIENT_ID = resolve_client_id(CLIENT_ID_OVERRIDE)
    client_id_project = "CLIENT_ID_OVERRIDE"
    client_id_values = {}
else:
    raw_client_id, client_id_project, client_id_values = get_secret_from_candidates("C17527K_MS")
    CLIENT_ID = resolve_client_id(raw_client_id)

CLIENT_SECRET, client_secret_project, client_secret_values = get_secret_from_candidates("C17527K_AD")

if not CLIENT_ID:
    raise RuntimeError("CLIENT_ID is empty. Check secret C17527K_MS.")

if not is_uuid(CLIENT_ID):
    raise RuntimeError(
        "CLIENT_ID is not a valid GUID. Check secret C17527K_MS or CLIENT_ID_OVERRIDE."
    )

if not CLIENT_SECRET:
    raise RuntimeError("CLIENT_SECRET is empty. Check secret C17527K_AD.")

print("Using authority:", AUTHORITY)
print("Using client_id:", CLIENT_ID)
print("CLIENT_ID source:", client_id_project)
print("CLIENT_SECRET source:", client_secret_project)

if client_id_values and len({v for v in client_id_values.values() if v}) > 1:
    print("WARNING: C17527K_MS differs across projects:")
    for project_id, value in client_id_values.items():
        if value:
            print(f"  {project_id}: {resolve_client_id(value)}")

if client_secret_values and len({v for v in client_secret_values.values() if v}) > 1:
    print("WARNING: C17527K_AD differs across projects (values hidden).")
    for project_id, value in client_secret_values.items():
        if value:
            print(f"  {project_id}: [present]")

Acquire Access Token

In [ ]:
from msal import ConfidentialClientApplication

app = ConfidentialClientApplication(
    client_id=CLIENT_ID,
    authority=AUTHORITY,
    client_credential=CLIENT_SECRET,
    )

token = app.acquire_token_for_client(
    scopes=["https://graph.microsoft.com/.default"]
    )

if "access_token" not in token:
    error = token.get("error", "")
    details = token.get("error_description", "")
    if error == "unauthorized_client" and "AADSTS700016" in details:
        raise RuntimeError(
            "MSAL auth failed: AADSTS700016. The CLIENT_ID is not an app registration in this tenant. "
            "Update secret C17527K_MS to the correct App (client) ID for tenant "
            f"{TENANT_ID}. Current CLIENT_ID={CLIENT_ID}."
        )
    raise RuntimeError(f"MSAL auth failed: {error} - {details}")

access_token = token["access_token"]
print("Access token acquired.")

Resolve SharePoint file URL via Microsoft Graph

In [ ]:
# blog calendar
# https://bcbst.sharepoint.com/:x:/r/sites/ID-DataScienceCOE/_layouts/15/Doc.aspx?sourcedoc=%7B2022496A-D803-48A2-8942-ED313EC883B9%7D&file=Blog_Calendar_Sign-up.xlsx&action=default&mobileredirect=true

sharepoint_file_url = (
    "https://bcbst.sharepoint.com/:x:/r/sites/ID-DataScienceCOE/_layouts/15/Doc.aspx?sourcedoc=%7B2022496A-D803-48A2-8942-ED313EC883B9%7D&file=Blog_Calendar_Sign-up.xlsx&action=default&mobileredirect=true"
    )

# Remove query params if present
clean_url = sharepoint_file_url.split("?")[0]

Encode URL for Graph /shares endpoint

In [ ]:
encoded_url = quote(clean_url, safe="")
share_id = f"u!{encoded_url.encode('utf-8').hex()}"


Download file bytes from Graph (no disk writes)

In [ ]:
headers = {
    "Authorization": f"Bearer {access_token}"
}

# Resolve metadata + download URL
meta_resp = requests.get(
    f"https://graph.microsoft.com/v1.0/shares/{share_id}",
    headers=headers,
)
meta_resp.raise_for_status()

download_url = meta_resp.json()["@microsoft.graph.downloadUrl"]

# Download the actual file bytes
file_bytes = requests.get(download_url).content

Load Excel directly into pandas

In [ ]:
df = pd.read_excel(BytesIO(file_bytes))
display(df)